In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_file(filename):
    candidates = [
        Path("data/raw") / filename,
        Path("../data/raw") / filename,
        Path(filename)
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"{filename} not found in expected locations.")

def ensure_output_dir(path_str):
    p = Path(path_str)
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


events_path = find_file("Events.csv")
fighters_stats_path = find_file("Fighters Stats.csv")
fighters_path = find_file("Fighters.csv")
fights_path = find_file("Fights.csv")

events = pd.read_csv(events_path)
fighters_stats = pd.read_csv(fighters_stats_path)
fighters = pd.read_csv(fighters_path)
fights = pd.read_csv(fights_path)

print("Loaded:")
print("Events:", events.shape)
print("Fighters Stats:", fighters_stats.shape)
print("Fighters:", fighters.shape)
print("Fights:", fights.shape)


def clean_text(series):
    return (
        series.astype(str)
        .str.strip()
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
    )

def to_numeric_safe(series):
    return pd.to_numeric(series, errors="coerce")

def first_existing(df, cols):
    existing = [c for c in cols if c in df.columns]
    if not existing:
        return pd.Series([np.nan] * len(df), index=df.index)
    out = df[existing[0]].copy()
    for c in existing[1:]:
        out = out.combine_first(df[c])
    return out




events = events.drop_duplicates().copy()
if "Date" in events.columns:
    events["Date"] = pd.to_datetime(events["Date"], errors="coerce")
if "Location" in events.columns:
    events["Location"] = clean_text(events["Location"])


fighters_stats = fighters_stats.drop_duplicates().copy()
for col in ["Full Name", "Nickname", "Stance", "Weight_Class", "Gender", "Fighting Style"]:
    if col in fighters_stats.columns:
        fighters_stats[col] = clean_text(fighters_stats[col])


fighters = fighters.drop_duplicates().copy()
for col in ["Full Name", "Nickname", "Stance"]:
    if col in fighters.columns:
        fighters[col] = clean_text(fighters[col])


fights = fights.drop_duplicates().copy()
for col in ["Weight_Class", "Time Format", "Result_1", "Result_2", "Referee", "Method", "Method Details"]:
    if col in fights.columns:
        fights[col] = clean_text(fights[col])


if "Result_1" in fights.columns:
    fights = fights[fights["Result_1"].isin(["W", "L"])].copy()


fights["target"] = (fights["Result_1"] == "W").astype(int)


clean_fighters = fighters.merge(
    fighters_stats,
    on="Fighter_Id",
    how="outer",
    suffixes=("_fighters", "_stats")
)

clean_fighters_final = pd.DataFrame({
    "Fighter_Id": clean_fighters["Fighter_Id"],
    "Full_Name": first_existing(clean_fighters, ["Full Name_stats", "Full Name_fighters"]),
    "Nickname": first_existing(clean_fighters, ["Nickname_stats", "Nickname_fighters"]),
    "Height": first_existing(clean_fighters, ["Ht._stats", "Ht._fighters"]),
    "Weight": first_existing(clean_fighters, ["Wt._stats", "Wt._fighters"]),
    "Reach": first_existing(clean_fighters, ["Reach"]),
    "Stance": first_existing(clean_fighters, ["Stance_stats", "Stance_fighters"]),
    "Belt": first_existing(clean_fighters, ["Belt_stats", "Belt_fighters"]),
    "Gender": first_existing(clean_fighters, ["Gender"]),
    "Weight_Class_Profile": first_existing(clean_fighters, ["Weight_Class"]),


    "W": first_existing(clean_fighters, ["W_stats", "W_fighters"]),
    "L": first_existing(clean_fighters, ["L_stats", "L_fighters"]),
    "D": first_existing(clean_fighters, ["D_stats", "D_fighters"]),
    "Avg_Fight_Time": first_existing(clean_fighters, ["Avg Fight Time"]),
    "KD": first_existing(clean_fighters, ["KD"]),
    "STR": first_existing(clean_fighters, ["STR"]),
    "TD": first_existing(clean_fighters, ["TD"]),
    "SUB": first_existing(clean_fighters, ["SUB"]),
    "Ctrl": first_existing(clean_fighters, ["Ctrl"]),
    "Sig_Str_Pct": first_existing(clean_fighters, ["Sig. Str. %"]),
    "Sub_Att": first_existing(clean_fighters, ["Sub. Att"]),
    "KO_Rate": first_existing(clean_fighters, ["KO Rate"]),
    "SUB_Rate": first_existing(clean_fighters, ["SUB Rate"]),
    "DEC_Rate": first_existing(clean_fighters, ["DEC Rate"]),
    "Fighting_Style": first_existing(clean_fighters, ["Fighting Style"]),
})


for col in ["Full_Name", "Nickname", "Stance", "Gender", "Weight_Class_Profile", "Fighting_Style"]:
    clean_fighters_final[col] = clean_text(clean_fighters_final[col])


numeric_cols_fighters = [
    "Height", "Weight", "Reach", "W", "L", "D", "Avg_Fight_Time",
    "KD", "STR", "TD", "SUB", "Ctrl", "Sig_Str_Pct",
    "Sub_Att", "KO_Rate", "SUB_Rate", "DEC_Rate"
]
for col in numeric_cols_fighters:
    clean_fighters_final[col] = to_numeric_safe(clean_fighters_final[col])


clean_fighters_final = clean_fighters_final.drop_duplicates(subset=["Fighter_Id"]).copy()


safe_fight_cols = [
    "Fight_Id",
    "Fighter_Id_1",
    "Fighter_Id_2",
    "Fighter_1",
    "Fighter_2",
    "Weight_Class",
    "Event_Id",
    "Result_1",
    "Result_2",
    "Time Format",
    "Referee",
    "target"
]
safe_fight_cols = [c for c in safe_fight_cols if c in fights.columns]

clean_fights = fights[safe_fight_cols].copy()


rename_map = {}
if "Fighter_1" in clean_fights.columns:
    rename_map["Fighter_1"] = "Raw_Fighter_1"
if "Fighter_2" in clean_fights.columns:
    rename_map["Fighter_2"] = "Raw_Fighter_2"
clean_fights = clean_fights.rename(columns=rename_map)


event_cols = [c for c in ["Event_Id", "Date", "Location", "Name"] if c in events.columns]

model_base = clean_fights.merge(
    events[event_cols],
    on="Event_Id",
    how="left"
)


fighter_cols = [c for c in clean_fighters_final.columns if c != "Fighter_Id"]

f1 = clean_fighters_final.rename(columns={col: f"f1_{col}" for col in fighter_cols})
f1 = f1.rename(columns={"Fighter_Id": "Fighter_Id_1"})

model_base = model_base.merge(
    f1,
    on="Fighter_Id_1",
    how="left"
)


f2 = clean_fighters_final.rename(columns={col: f"f2_{col}" for col in fighter_cols})
f2 = f2.rename(columns={"Fighter_Id": "Fighter_Id_2"})

model_base = model_base.merge(
    f2,
    on="Fighter_Id_2",
    how="left"
)


model_base["Fighter_1_Name"] = model_base["f1_Full_Name"]
model_base["Fighter_2_Name"] = model_base["f2_Full_Name"]


if "Raw_Fighter_1" in model_base.columns and "Raw_Fighter_2" in model_base.columns:
    model_base["raw_name_match_flag"] = (
        (model_base["Raw_Fighter_1"] == model_base["Fighter_1_Name"]) &
        (model_base["Raw_Fighter_2"] == model_base["Fighter_2_Name"])
    ).astype(int)


num_pairs = [
    ("Height", "height_diff"),
    ("Weight", "weight_diff"),
    ("Reach", "reach_diff"),
    ("W", "wins_diff"),
    ("L", "losses_diff"),
    ("D", "draws_diff"),
    ("Avg_Fight_Time", "avg_fight_time_diff"),
    ("KD", "kd_diff"),
    ("STR", "str_diff"),
    ("TD", "td_diff"),
    ("SUB", "sub_diff"),
    ("Ctrl", "ctrl_diff"),
    ("Sig_Str_Pct", "sig_str_pct_diff"),
    ("Sub_Att", "sub_att_diff"),
    ("KO_Rate", "ko_rate_diff"),
    ("SUB_Rate", "sub_rate_diff"),
    ("DEC_Rate", "dec_rate_diff"),
]

for base_col, diff_col in num_pairs:
    c1 = f"f1_{base_col}"
    c2 = f"f2_{base_col}"
    if c1 in model_base.columns and c2 in model_base.columns:
        model_base[c1] = to_numeric_safe(model_base[c1])
        model_base[c2] = to_numeric_safe(model_base[c2])
        model_base[diff_col] = model_base[c1] - model_base[c2]


repo_root = events_path.resolve().parents[2]

interim_dir = repo_root / "data" / "interim"
processed_dir = repo_root / "data" / "processed"

interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

clean_events_out = interim_dir / "clean_events.csv"
clean_fighters_out = interim_dir / "clean_fighters.csv"
clean_fights_out = interim_dir / "clean_fights.csv"
model_base_out = processed_dir / "model_base_table.csv"

events.to_csv(clean_events_out, index=False)
clean_fighters_final.to_csv(clean_fighters_out, index=False)
clean_fights.to_csv(clean_fights_out, index=False)
model_base.to_csv(model_base_out, index=False)

print("\nSaved:")
print(f"- {clean_events_out.resolve()}")
print(f"- {clean_fighters_out.resolve()}")
print(f"- {clean_fights_out.resolve()}")
print(f"- {model_base_out.resolve()}")

print("\nExist check:")
print("clean_events.csv exists:", clean_events_out.exists())
print("clean_fighters.csv exists:", clean_fighters_out.exists())
print("clean_fights.csv exists:", clean_fights_out.exists())
print("model_base_table.csv exists:", model_base_out.exists())


print("\nSaved:")
print(f"- {clean_events_out}")
print(f"- {clean_fighters_out}")
print(f"- {clean_fights_out}")
print(f"- {model_base_out}")

print("\nModel base shape:", model_base.shape)

important_nulls = model_base[
    [c for c in [
        "Date", "f1_Full_Name", "f2_Full_Name",
        "f1_Height", "f2_Height", "f1_Reach", "f2_Reach"
    ] if c in model_base.columns]
].isnull().mean().sort_values(ascending=False) * 100

print("\nImportant missing percentages:")
print(important_nulls.round(2))

if "raw_name_match_flag" in model_base.columns:
    print("\nRaw name match ratio:")
    print((model_base["raw_name_match_flag"].mean() * 100).round(2), "%")

print("\nPreview:")
preview_cols = [c for c in [
    "Fight_Id", "Fighter_Id_1", "Fighter_Id_2",
    "Raw_Fighter_1", "Raw_Fighter_2",
    "Fighter_1_Name", "Fighter_2_Name",
    "Weight_Class", "target"
] if c in model_base.columns]

print(model_base[preview_cols].head())

Loaded:
Events: (756, 4)
Fighters Stats: (2602, 37)
Fighters: (4448, 11)
Fights: (8461, 47)

Saved:
- C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\interim\clean_events.csv
- C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\interim\clean_fighters.csv
- C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\interim\clean_fights.csv
- C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\processed\model_base_table.csv

Exist check:
clean_events.csv exists: True
clean_fighters.csv exists: True
clean_fights.csv exists: True
model_base_table.csv exists: True

Saved:
- C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\interim\clean_events.csv
- C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\interim\clean_fighters.csv
- C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\interim\clean_fights.csv
- C:\Users\Victus\Des

In [5]:
import pandas as pd
from pathlib import Path
import os

print("Current working directory:", os.getcwd())


input_path = Path("../data/processed/model_base_table.csv")

if not input_path.exists():
    raise FileNotFoundError(f"File not found: {input_path.resolve()}")

df = pd.read_csv(input_path, low_memory=False)


df = df.dropna(subset=["Fighter_1_Name", "Fighter_2_Name"]).copy()


cols_to_drop = ["Raw_Fighter_1", "Raw_Fighter_2", "raw_name_match_flag"]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)


df = df.rename(columns={
    "Fighter_1_Name": "Fighter_1",
    "Fighter_2_Name": "Fighter_2"
})


df = df.loc[:, ~df.columns.duplicated()]


output_path = Path("../data/processed/model_base_table_clean_names.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print("Saved to:", output_path.resolve())
print("Shape:", df.shape)

preview_cols = ["Fight_Id", "Fighter_Id_1", "Fighter_1", "Fighter_Id_2", "Fighter_2"]
preview_cols = [c for c in preview_cols if c in df.columns]
print(df[preview_cols].head())

Current working directory: C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\notebooks
Saved to: C:\Users\Victus\Desktop\DSA210project\Fightiq-ufc-matchup-intelligence\data\processed\model_base_table_clean_names.csv
Shape: (8310, 80)
           Fight_Id      Fighter_Id_1       Fighter_1      Fighter_Id_2  \
0  4a0db214d9721d6e  d661ce4da776fc20        Petr Yan  c03520b5c88ed6b4   
1  dfa692db6d39330c  17e97649403ba428      Joshua Van  a0f0004aadf10b71   
2  fbbb9e72900b71f5  4461d7e47375a895   Tatsuro Taira  792be9a24df82ed6   
3  1dc29f4c6fcdd356  6e743a33d56bdaa4  Payton Talbott  056c493bbd76a918   
4  c797b58d85c840a4  2e7878927067fdca   Manuel Torres  99bd51917728c25d   

           Fighter_2  
0  Merab Dvalishvili  
1  Alexandre Pantoja  
2     Brandon Moreno  
3       Henry Cejudo  
4       Grant Dawson  
